# EDA — Prédiction HPP sévère (synthèse)

Notebook **pro** pour la certification. Le détail exploratoire est archivé dans `01_EDA_exploration.ipynb`.

**Objectif** : prédire le risque d'hémorragie du post-partum sévère (`hpp_trans`) **avant l'accouchement** (anti data leakage).

**Priorité métier** : maximiser le **rappel** (ne pas rater une patiente à risque).


## 1. Chargement des données

Les données brutes (≈ 60k lignes) restent hors repo (RGPD / volume) — voir `../00_Data/DATA_LOCATION.md`.
Ici on utilise l'extrait portfolio + le dictionnaire de variables.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("..") / "00_Data"
extract = DATA / "extract_database.csv"
dico = DATA / "dico_var.csv"

df = pd.read_csv(extract, low_memory=False)
desc = pd.read_csv(dico, index_col=0) if dico.exists() else None
print(f"Extrait : {df.shape[0]} lignes × {df.shape[1]} colonnes")
if "hpp_trans" in df.columns:
    print(df["hpp_trans"].value_counts(normalize=True).rename("part").round(4))
df.head(3)


## 2. Périmètre : variables pré-accouchement

Seules les variables **disponibles avant l'accouchement** sont retenues (cohérence temporelle, pas de fuite d'info).
Liste finale alignée sur le modèle déployé (Streamlit / API).


In [ ]:
# Features du modèle en production (joblib Streamlit / API)
FEATURES_FINAL = [
    "bmi", "terme", "dsm_g", "taille_mere", "age_m", "parite", "nbilan",
    "hosp_m_g", "nsej18", "AMP", "g_type", "diabete", "preecl", "hta_tot",
    "tabac", "cortico", "ut_cica", "creta", "hellp", "cholestase", "pma", "bilan",
]
TARGET = "hpp_trans"

available = [c for c in FEATURES_FINAL if c in df.columns]
missing_cols = [c for c in FEATURES_FINAL if c not in df.columns]
print("Features dispo dans l'extrait :", len(available), "/", len(FEATURES_FINAL))
if missing_cols:
    print("Absentes de l'extrait (présentes en base complète) :", missing_cols)

use_cols = available + ([TARGET] if TARGET in df.columns else [])
eda = df[use_cols].copy()
eda.head(3)


## 3. Qualité des données (résumé)


In [ ]:
quality = pd.DataFrame({
    "dtype": eda.dtypes.astype(str),
    "n_missing": eda.isna().sum(),
    "pct_missing": (eda.isna().mean() * 100).round(2),
    "n_unique": eda.nunique(dropna=True),
}).sort_values("pct_missing", ascending=False)
quality


## 4. Cible `hpp_trans` — déséquilibre de classes

≈ 2 % de positifs en base complète → défi de modélisation (SMOTE / class weight testés ensuite).


In [ ]:
if TARGET in eda.columns:
    counts = eda[TARGET].value_counts()
    fig, ax = plt.subplots(figsize=(5, 3))
    counts.plot(kind="bar", ax=ax, color=["#1B2A4A", "#B03A2E"])
    ax.set_title("Distribution hpp_trans (extrait)")
    ax.set_xlabel("Classe")
    ax.set_ylabel("Effectif")
    plt.tight_layout()
    plt.show()
    print((counts / counts.sum() * 100).round(2).astype(str) + " %")
else:
    print("Colonne cible absente de l'extrait")


## 5. Décisions de features (issus de l'EDA exploratoire)

| Écartée | Motif |
|---------|--------|
| `type_grossesse` | Colinéaire avec `g_type` (moins de NA) |
| `parite_cor`, `bas_risque` | Peu informatives / déséquilibrées |
| `Aide_procreation` | Colinéaire avec `AMP` |
| `Dosecortico` | Trop de NA vs `cortico` |
| `sej18` | Colinéaire avec `nsej18` |
| `poids_mere` | Colinéaire avec `bmi` |
| `hta_gest`, `hta_chro` | Redondantes vs `hta_tot` |

Nettoyage notable : `dsm_g` borné dans `[0, 270]` ; 1 NA sur `age_m` droppé en base complète.

Détail des tests (Kendall, χ², matrices) → **`01_EDA_exploration.ipynb`**.


## 6. Suite du pipeline

1. `02_Modelisation_imputation.ipynb` — stratégies NA
2. `03` / `04` / `05` — LogReg, RF, XGBoost + resampling
3. Artefact prod : `best_model_logreg_f1_Sans_resampling.joblib`
4. Industrialisation : **API FastAPI** (`../app/`) + Streamlit + MLflow
